## Simulation

This notebook computes historic time series for implied vols, fitting error, DLV vs a gixed NMS grid.



In [ ]:
%load_ext autoreload
%autoreload 1

%aimport quantdata
%aimport quantdata.biz_calendar
%aimport quantdata.widb
%aimport quantdata.widb.client
%aimport quantdata.widb.day
%aimport quantdata.widb.arbfree
%aimport quantdata.widb.options_filter
%aimport quantdata.dysanos
%aimport quantdata.utils.backbone
%aimport quantdata.sanos.sanos_tools
%aimport quantdata.sanos.arbfree
%aimport quantdata.sanos.sanos
%aimport quantdata.dysanos.grid.torch_market
%aimport quantdata.dysanos.grid.torch_pricer
%aimport quantdata.dysanos.grid.torch_model
%aimport quantdata.dysanos.grid.torch_gfx 
%aimport quantdata.dysanos.grid.torch_train 
%aimport quantdata.dysanos.grid.torch_simulator

from quantdata.dysanos.grid.torch_market import GridMarketDefinition, DateRange, PoolConfig, Context, date, GridConfig

grid_config     = GridConfig(grid_num_strikes=100)
grid_market_def = GridMarketDefinition( ticker="SPX", date_range=DateRange(start=date(2020, 1, 1), end=date(2025, 8, 30)), grid_config=grid_config )
market          = grid_market_def.load( pool_config = PoolConfig(num_workers=4, threading=False ), verbose=Context.all )
print(f"nms /done - found {len(market['dates'])} dates.")
    

In [ ]:
from cdxcore.dynaplot import FigStore, colors_tableau, figure, m_o_m
from quantdata.pool import PoolConfig
from quantdata.dysanos.grid.torch_gfx import VizConfig
from quantdata.dysanos.grid.torch_train import train, TrainConfig
from quantdata.dysanos.grid.torch_model import BaseEnvironment, FeedForward, SanosModel, SanosModelConfig, SanosDecoderConfig
import numpy as np

mdl_config = SanosModelConfig( 
    label="ac_state20",
    user_version="0.2",
    group="sm25_100",
    encoder_config = FeedForward.Config(
            activation = "SELU",
            num_depth = 2,
            num_width = 100,
            use_bias = True,
        ),
    decoder_config = SanosDecoderConfig(
        smoothness = 0.25,
        num_state = 20, 
        nn_config = FeedForward.Config(
            activation = "SELU",
            num_depth = 2,
            num_width = 100,
            use_bias = True,
        ),
    ),
)

env          = BaseEnvironment()
viz_config   = VizConfig( num_viz_dates=4 )
trn_config   = TrainConfig( num_epochs=100_000, lr=1E-4, final_lr=1E-6, weight_decay=1e-4, batch_size=400, compile=True, epoch_interval=10000 )
model        = train( market, mdl_config=mdl_config, trn_config=trn_config, viz_config=viz_config, just_load=True, env=env )
torch_data   = mdl_config.torch_data( env = env, market = market, verbose = Context.quiet )
model_output = model( torch_data, training=False )
print("/done")

## Visualize states

In [ ]:
from quantdata.dysanos.grid.torch_simulator import ARSamplerConfig, ARSampler, Context
from cdxcore.dynaplot import figure, colors_tableau, m_o_m
import numpy as np
from IPython.display import Markdown, display

ar_config  = ARSamplerConfig(num_alpha=5, use_matrix_exp=True)
sampler    = ARSampler( market, model, torch_data, config=ar_config, verbose=Context.all )

def plot():
    dates     = market.dates
    num_alpha = sampler.num_alpha

    with figure(row_size=4, col_size=6, columns=6 ) as fig:
        display(Markdown(f"## {model.qualified_label} real and simulated states, discrete to normal randomness, for {ar_config.num_alpha} PCA factors"))

        r = sampler.pcaxar1
        xstate = sampler.xstate
        ndt    = r.ndt[:,None]
        dX     = ( r.U0 - sampler.xstate[:-1,:] @ r.k ) * ndt + r.alpha @ r.dU * np.sqrt(ndt)
        recX   = np.concatenate( [ sampler.xstate[:1,:], sampler.xstate[:1,:] + np.cumsum( dX, axis=0 ) ], axis=0 )
        assert recX.shape == xstate.shape, f"{recX.shape} != {xstate.shape}"

        rec        = np.zeros((len(dates), xstate.shape[1], 5))
        rec[...,0] = xstate
        rec[...,1] = sampler.sample_xstates( index=0, num_steps=len(dates), num_samples=0, bandwidth=0., replay=True )
        rec[...,2] = sampler.sample_xstates( index=0, num_steps=len(dates), num_samples=0, bandwidth=0.1, replay=False )
        rec[...,3] = sampler.sample_xstates( index=0, num_steps=len(dates), num_samples=0, bandwidth=0.5, replay=False )
        rec[...,4] = sampler.sample_xstates( index=0, num_steps=len(dates), num_samples=0, bandwidth=1., replay=False )

        labels = ["Market", "0%", "10%", "50%", "100%"]
        
        ax_ = None
        ax = fig.add_subplot(f"State 0 (log-spot)")
        for j in range(rec.shape[-1]):
            ax.plot( dates, rec[:,0,j], "o" if j==0 else "-", label=labels[j])
        ax.tick_params(axis="x", labelrotation=90)
        ax.legend(loc="upper right")
        for i in range(1,xstate.shape[1]):
            ax = fig.add_subplot(f"State {i}")
            for j, label in enumerate(labels):
                ax.plot( dates, rec[:,i,j], "o" if j==0 else "-", )
            ax.tick_params(axis="x", labelrotation=90)
    
    with figure(row_size=4, col_size=6, columns=6 ) as fig:
        display(Markdown(f"## {model.qualified_label} factors"))
        U0   = sampler.pcaxar1.U0
        dU   = sampler.pcaxar1.dU
        colors = colors_tableau()
        x    = np.arange(U0.shape[1])
        ax0  = fig.add_subplot("U0")
        ax0.bar( x, U0[0,:], width=0.2, color="black")
        for i in range( dU.shape[0] ):
            ax = fig.add_subplot(f"dU[{i}]")
            ax.bar( x, dU[i,:], width=0.2, color=next(colors))
            ax.sharex(ax0)
        ax0.set_ylim( m_o_m( U0, dU ) )
    
    with figure(row_size=4, col_size=6, columns=6 ) as fig:
        display(Markdown(f"## {model.qualified_label} weights"))
        mev = sampler.pcaxar1.vars_explained
        
        ax = fig.add_subplot("Explained")
        ax.plot( np.arange(0,len(mev)), mev, ":", color="orange", label="All")
        ax.plot( np.arange(0,num_alpha), mev[:num_alpha], "-", color="red",label=f"First {num_alpha}")
        ax.set_xlabel("% of total variance explained")
        ax.set_ylabel("Eigenvalue")
        
        ax_ = None
        colors = colors_tableau()
        alpha  = sampler.pcaxar1.alpha
        
        for i in range(alpha.shape[1]):
            ax = fig.add_subplot(f"Alpha {i}")
            ax.plot( dates[:-1], alpha[:,i], "*:", label="Model", color=next(colors))
            ax.tick_params(axis="x", labelrotation=90)
            if not ax_ is None:
                ax.sharey(ax_)
            else:
                ax_ = ax
        ax_.set_ylim( m_o_m(alpha) )

    
plot()




## Show some path from t to t+1

In [ ]:
from dataclasses import dataclass
from cdxcore.err import verify_inp
from cdxcore.verbose import Context
from cdxcore.pretty import PrettyObject
from quantdata.dysanos.grid.torch_simulator import ARSamplerConfig, ARSampler
from quantdata.biz_calendar import DAYS_PER_YEAR
from quantdata.torch import BaseEnvironment
from cdxcore.dynaplot import figure, FigStore
import torch
import numpy as np
from cdxcore.bs import bs
import time

def plot_vol( ax, expiries, strikes, vols, *, store = None, first = True ):
    exp_block = np.stack( [ expiries*DAYS_PER_YEAR ]*(strikes.shape[-1]), axis=-1 )
    exp_range = (expiries.max()*DAYS_PER_YEAR, expiries.min()*DAYS_PER_YEAR)
    
    r = ax.plot_surface( exp_block, strikes, vols, cmap='viridis', edgecolor='none')
    if not store is None:
        store += r
    if first:
        ax.set_xlabel("Days")
        ax.set_ylabel("K")
        ax.view_init(elev=25, azim=45) 
        ax.set_xlim(*exp_range)
        ax.set_ylim( strikes.min(), strikes.max() )

def simulate_1( indices, num_samples=4, num_steps=2, loop=False ):
    env            = BaseEnvironment()
    ns_config      = ARSamplerConfig(num_alpha=20)
    normal_sampler = ARSampler( model=model, market=market, torch_data=torch_data, config=ns_config )

    figures   = []
    sim_axs   = {}
    store     = FigStore()
    orig_vols = {}

    markets   = []
    
    def vols_for_state( state, index_tdata, expiries, strikes ):
        assert len(state.shape) == 1
        index_state        = torch.tensor( state[None,:], device=env.device, dtype=env.fdtype ) if isinstance(state, np.ndarray) else state[None,:]
        index_model_output = model( index_tdata, state=index_state, training=False )
        mdl_calls          = index_model_output['fits'][0,...].detach().cpu().numpy()
        return bs.implied( k=strikes, sqrtT=np.sqrt(expiries)[:,None], prices=mdl_calls, on_exceed_bounds="quiet" )
        
    for index in indices:
        assert index < len(market.dates)-1
        index2       = index+num_steps-1
        t_market     = { k: v[index:index+1,...] if isinstance(v, np.ndarray) else v for k,v in market.items() }
        tp1          = PrettyObject()
        tp1.torch_data    = { k: v[index2:index2+1,...] for k,v in torch_data.items() }
        tp1.market   = { k: v[index2:index2+1,...] if isinstance(v, np.ndarray) else v for k,v in market.items() }
        tp1.expiries = tp1.market['expiries'][0,:]
        tp1.strikes  = tp1.market['xstrikes'][0,:,1:-1]

        display(Markdown(f"## {market.ticker} sample #{index} {market.dates[index]}->{market.dates[index2]}"))

        fig = figure(fig_size=(40,10), columns=2+num_samples )
        figures.append( fig )
        gs  = fig.add_gridspec(2,2+num_samples)

        ax_t = fig.add_subplot(f"Day t {market.dates[index]}", projection='3d', spec_pos=gs[0,0] )
        vol_t = t_market['mid_vols'][0,:,:] 
        plot_vol( ax_t, t_market['expiries'][0,:], t_market['xstrikes'][0,:,1:-1], vol_t )

        ax_tp1 = fig.add_subplot(f"Day t+1 {market.dates[index+1]} Actual", projection='3d', spec_pos=gs[0,1] )
        tp1.mid_vols = tp1.market['mid_vols'][0,:,:]
        plot_vol( ax_tp1, tp1.expiries, tp1.strikes, tp1.mid_vols )

        mom = m_o_m( vol_t, tp1.mid_vols )
        ax_t.set_zlim( mom )
        ax_tp1.set_zlim( mom )

        ax_ref  = fig.add_subplot( f"Diff PCA ref for {market.dates[index+1]}", projection='3d', spec_pos=gs[1,0] )
        vol_ref = vols_for_state( normal_sampler.sample_xstates( index, num_steps=2, bandwidth=0., replay=True )[1,1:], 
                                  index_tdata=tp1.torch_data, expiries=tp1.expiries, strikes=tp1.strikes )
        plot_vol( ax_ref, tp1.expiries, tp1.strikes, vol_ref - tp1.mid_vols )
        ax_ref.set_zlim( -0.05, 0.05 )

        ax_diff_act  = fig.add_subplot( f"Diff vol(t+1) - vol(t), actual", projection='3d', spec_pos=gs[1,1] )
        plot_vol( ax_diff_act, tp1.expiries, tp1.strikes, tp1.mid_vols - vol_t )
        ax_diff_act.set_zlim( -0.05, 0.05 )

        sample_axs = []
        for isample in range(num_samples):
            sample_ax = fig.add_subplot( f"Sample #{isample} for {market.dates[index+1]}", projection='3d', spec_pos=gs[0,2+isample] )
            
            diff_ax   = fig.add_subplot( f"Diff vol(t+1) - vol(t) sample #{isample}", projection='3d', spec_pos=gs[1,2+isample] )
            sample_ax.set_zlim( mom )
            diff_ax.set_zlim( -0.05, 0.05 )
            sample_axs.append( (sample_ax, diff_ax) )

        sim_axs[index] = fig, sample_axs, tp1, vol_t
        fig.render()

    # simulate
    # --------

    try:
        first = True
        while True:                    
            store.clear()
            for index, (fig, sample_axs, tp1, vol_t) in sim_axs.items():
                new_states  = normal_sampler.sample_log_nspot_states( index=index, num_samples=num_samples, num_steps=num_steps, bandwidth=0.2, replay=False )[1][:,-1,:]
                assert new_states.shape == (num_samples, model.config.num_state), f"Internal error {new_states.shape} != {(num_samples, model.config.num_state)}"
                new_states  = torch.tensor( new_states, dtype=env.fdtype, device=env.device )

                mom_v = None
                mom_d = None
                for isample, (sample_ax, diff_ax) in enumerate(sample_axs):
                    tp1_model_output = model( tp1.torch_data, state=new_states[isample:isample+1,:], training=False )
                    mdl_vols         = vols_for_state( new_states[isample,:], index_tdata=tp1.torch_data, expiries=tp1.expiries, strikes=tp1.strikes )
                    plot_vol( sample_ax, tp1.expiries, tp1.strikes, mdl_vols, store=store, first=first )
                    plot_vol( diff_ax, tp1.expiries, tp1.strikes, mdl_vols-vol_t, store=store, first=first )
                fig.render()
            first = False
            if not loop:
                break
            time.sleep(1)
        
    except KeyboardInterrupt as e:
        for fig in figures:
            fig.close()
        raise e

num_steps   = 6
num_indices = 2
num_samples = 8
simulate_1( indices = [ int(i) for i in np.linspace(0, len(market.dates)-num_steps, num_indices) ] if num_indices > 1 else [len(market.dates)-num_steps], 
            num_samples=num_samples,
            loop=False )
    

    
    


        

## Simulate N steps

In [ ]:
from dataclasses import dataclass
from cdxcore.err import verify_inp
from cdxcore.verbose import Context
from cdxcore.pretty import PrettyObject
from quantdata.dysanos.grid.torch_simulator import ARSamplerConfig, ARSampler
from quantdata.biz_calendar import DAYS_PER_YEAR
from quantdata.torch import BaseEnvironment
from cdxcore.dynaplot import figure, FigStore
import torch
import numpy as np
from cdxcore.bs import bs
import time
from IPython.display import Markdown, display

def plot_vol( ax, expiries, strikes, vols, *, store = None, first = True ):
    exp_block = np.stack( [ expiries*DAYS_PER_YEAR ]*(strikes.shape[-1]), axis=-1 )
    exp_range = (expiries.max()*DAYS_PER_YEAR, expiries.min()*DAYS_PER_YEAR)
    
    r = ax.plot_surface( exp_block, strikes, vols, cmap='viridis', edgecolor='none')
    if not store is None:
        store += r
    if first:
        ax.set_xlabel("Days")
        ax.set_ylabel("K")
        ax.view_init(elev=25, azim=45) 
        ax.set_xlim(*exp_range)
        ax.set_ylim( strikes.min(), strikes.max() )

def simulate_N( indices, num_steps=4, num_samples=1, num_viz_steps=None, replay=False, bandwidth=0. ):
    env            = BaseEnvironment()
    ns_config      = ARSamplerConfig(num_alpha=21, seed=211)
    normal_sampler = ARSampler( model=model, market=market, torch_data=torch_data, config=ns_config )
    num_viz_steps  = num_viz_steps if not num_viz_steps is None else num_steps

    figures   = []
    sim_axs   = {}
    store     = FigStore()
    orig_vols = {}
    markets   = []
    ticker    = market.ticker

    verify_inp( num_viz_steps <= num_steps, "'num_viz_steps' must be smaller or equal to 'num_steps'")
    if num_viz_steps >= num_steps:
        viz_steps = np.arange(1,num_steps+1) 
    else:        
        viz_steps = np.concatenate( [ np.linspace( 1, num_steps-1, num_viz_steps-1, dtype=np.int32 ), [num_steps] ] )
    viz_steps = [ int(i) for i in viz_steps ]
    
    def vols_for_state( state, index_tdata, expiries, strikes ):
        assert len(state.shape) == 1
        index_state        = torch.tensor( state[None,:], device=env.device, dtype=env.fdtype ) if isinstance(state, np.ndarray) else state[None,:]
        index_model_output = model( index_tdata, state=index_state, training=False )
        mdl_calls          = index_model_output['fits'][0,...].detach().cpu().numpy()
        return bs.implied( k=strikes, sqrtT=np.sqrt(expiries)[:,None], prices=mdl_calls, on_exceed_bounds="quiet" )
        
    for index in indices:
        assert index+num_steps < len(market.dates), f"Index {index} with {num_steps} exceeds {len(market.dates)}"
        display(Markdown(fr"## {market.ticker} {market.dates[index]} $\rightarrow$ {market.dates[index+num_steps]}" + ('' if num_viz_steps >= num_steps else f' (showing {num_viz_steps}/{num_steps} steps)') + (' using discrete sampling ' if not replay else ' using discrete replay ') + f"with normal bandwidth {bandwidth:.2f}" ))
        r            = PrettyObject()
        r.market     = { k: v[index:index+num_steps+1,...] if isinstance(v, np.ndarray) else v for k,v in market.items() }
        r.torch_data = { k: v[index:index+num_steps+1,...] for k,v in torch_data.items() }
        r.expiries   = market['expiries'][index:index+num_steps+1,:]
        r.strikes    = market['xstrikes'][index:index+num_steps+1,:,1:-1]
        r.mid_vols   = market['mid_vols'][index:index+num_steps+1,:,:]
        r.log_spot   = normal_sampler.log_nspot[index:index+num_steps]

        log_spot, new_states  = normal_sampler.sample_log_nspot_states( index=index, num_samples=num_samples, num_steps=num_steps, replay=replay, bandwidth=bandwidth )
        assert log_spot.shape == (num_samples, num_steps), f"Internal error {log_spot.shape} != {(num_samples, num_steps)}"
        assert new_states.shape == (num_samples, num_steps, model.config.num_state), f"Internal error {new_states.shape} != {(num_samples, num_steps, model.config.num_state)}"
        new_states  = torch.tensor( new_states, dtype=env.fdtype, device=env.device )

        # plot 
        # ----

        with figure(row_size=5, col_size=20, columns=1 ) as fig:

            ax = fig.add_subplot(f"{ticker} spot actual vs. simulated" )
            ax.plot( market.dates[index:index+num_steps], np.exp(r.log_spot), ".-", color="black", label="Actual")

            for isample in range(num_samples):
                ax.plot( market.dates[index:index+num_steps], np.exp( log_spot[isample,:] ), ".:", label=f"Sample {isample}")
            
            ax.tick_params(axis="x", labelrotation=90)
            ax.legend(loc="upper left")
            fig.next_row()
            
        with figure(row_size=3, col_size=4, columns=1+num_steps ) as fig:
            axes = []
            # plot start
            ax = fig.add_subplot(f"{ticker} actual {market.dates[index]}", projection='3d' )
            plot_vol( ax, r.expiries[0,:], r.strikes[0,:,:], r.mid_vols[0,:,:] )
            axes.append( ax )
            
            # plot original path
            for istep in viz_steps:
                ii = index+istep
                ax = fig.add_subplot(f"{ticker} actual {market.dates[ii]}", projection='3d' )
                plot_vol( ax, r.expiries[istep,:], r.strikes[istep,:,:], r.mid_vols[istep,:,:] )
                axes.append( ax )
            fig.next_row()
            min_ = np.min(r.mid_vols)
            max_ = np.max(r.mid_vols)
            mom = m_o_m( r.mid_vols, buf=None )

            # for each sample plot that path
            for isample in range(num_samples):
                # plot start
                ax = fig.add_subplot(f"{ticker} actual {market.dates[index]}", projection='3d' )
                plot_vol( ax, r.expiries[0,:], r.strikes[0,:,:], r.mid_vols[0,:,:] )
                axes.append( ax )

                # plot simulation
                for istep in viz_steps:
                    ii       = index+istep
                    ax       = fig.add_subplot(f"{ticker} simulated {market.dates[ii]} path #{isample}", projection='3d' )
                    mdl_vols = vols_for_state( new_states[isample,istep-1,:],
                                               index_tdata={ k: v[ii:ii+1,...] for k,v in torch_data.items() }, 
                                               expiries=r.expiries[istep,:],
                                               strikes=r.strikes[istep,:,:] )
                    
                    plot_vol( ax, r.expiries[istep,:], r.strikes[istep,:,:], mdl_vols )
                    axes.append( ax )
                    mom = m_o_m( mom, mdl_vols, buf=None )
                fig.next_row()

            mom = m_o_m( mom, pos_floor=0.01 )
            for ax in axes:
                ax.set_zlim( mom )
            fig.render()

num_indices   = 1
num_viz_steps = 6
num_steps     = 48
num_samples   = 4
indices       = [ int(i) for i in np.linspace(0, len(market.dates)-num_steps-1, num_indices) ] if num_indices > 1 else [len(market.dates)-num_steps-1]
simulate_N( indices=indices, num_samples=num_samples, num_steps=num_steps, num_viz_steps=num_viz_steps, replay=False, bandwidth=0.2 )


    

  

## Just plot a real path

In [ ]:
def print_lots_market_data( num_rows = 4, num_cols = 6 ):

    num_viz   = num_rows * num_cols
    num_dates = len(market.dates)
    indices   = np.concat([[0],np.linspace(1,num_dates-2,num_viz-2,dtype=np.int32),[num_dates-1]])

    display(Markdown(f"## {market.ticker} Implied Vols from {market.dates[0]} to {market.dates[-1]}"))

    r            = PrettyObject()
    r.expiries   = market['expiries'][indices,:]
    r.strikes    = market['xstrikes'][indices,:,1:-1]
    r.mid_vols   = market['mid_vols'][indices,:,:]
    
    with figure( row_size=3, col_size=4, columns=num_cols ) as fig:

        mom = m_o_m( r.mid_vols )
        for ii, index in enumerate(indices):    
            # plot start
            ax = fig.add_subplot(f"{market.dates[index]}", projection='3d' )
            plot_vol( ax, r.expiries[ii,:], r.strikes[ii,:,:], r.mid_vols[ii,:,:] )
            ax.set_zlim( mom )
        
print_lots_market_data()

    
    